# Session 4 - Control flow examples and notes on file handing

File handling in programming refers to the process of creating, reading, writing, and manipulating files on a computer. It allows programs to store data persistently, retrieve it when needed, and manage file operations efficiently. The key function in python for file handling is the `open()` function, which is used to open a file in different modes such as read, write, append, etc. The Python documentation provides a comprehensive guide on file handling, including examples and best practices at the following url: [Python File Handling Documentation](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files)

`open()` returns a file object, and is most commonly used with two positional arguments and one keyword argument: open(filename, mode, encoding=None). The `filename` is the name of the file you want to open, `mode` specifies the mode in which the file is opened (e.g., 'r' for reading, 'w' for writing, 'a' for appending), and `encoding` specifies the encoding format (e.g., 'utf-8').




### STEP 1: `for-in` loop on file iterator to identify ports and their respective total FOB

This section explains the design of Step 1 in some detail AND it introduces Mermaid diagraming for illustrating the Control Flow implemented by the design.

We want to find out which ports handle the most coffee exports. So, our strategy will be to scan the raw records in our data file 'trase_global_coffee_supply_chain.csv' and to calculate the FOB totals per export port. 

```python
import csv

file_path = 'trase_global_coffee_supply_chain.csv'

port_fob_totals = {} 
# Define an empty dictionary to store the Ports by name and their respective total FOB. We can use a dictionary to store the port names as keys and their corresponding total FOB values as values. This allows us to efficiently accumulate the FOB values for each port as we iterate through the CSV file.

with open(file_path, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    # Iterate through each row in the CSV file. because reader is an iterable, the for loop will iterate over each row until all rows have been processed.

    for row in reader: 
    # this is a pre-test loop, the condition is checked first before executing the loop action. If the condition evaluates to false, the loop exits and the workflow continues to the subsequent process.

        # Extract port and fob from the current row using their respective column labels
        port = row['port_of_export_name']
        fob = row['fob']
        if port and fob: # This checks if both port and fob are both present, if not, skip to the next row
            port_fob_totals[port] = port_fob_totals.get(port, 0.0) + float(fob) 
            
            # The get() method is used to retrieve the current total FOB for the current port. Using port as its key, if the port_fob_totals dictionary entry for [port] has not been encountered before (i.e., it is not found in the dictionary), it initializes it's value to 0.0 and adds the current FOB value.. The float(fob) converts the FOB value from a string to a float for accurate numerical calculations.
            # And so we accumulate the FOB value for the corresponding port in the dictionary. 

print("=== STEP 1: PORT FOB TOTALS ===")
for port_name, total_fob in port_fob_totals.items(): 
# We can infer that port_name and total_fob are variables that represent the key-value pairs in the port_fob_totals dictionary, accessed using the .items() method on port_fob_totals. Because port_fob_totals is iterable, the for loop can iterate over it and return each key-value pair in the dictionary, allowing us to access both the port name and its corresponding total FOB value for printing.

    print(f"Port: {port_name:<30} | Total FOB: ${total_fob:,.2f}") 
    # We can use print's f-string structure to print the name of each port (limiting the string printed to 30 characters) and its corresponding total fob.
```

The Mermaid diagram code is rendered in the notebook if Matt Bierner's Markdown Preview Mermaid Support is installed in your VS Code.

```mermaid
flowchart TD
    Start([Start]) --> OpenFile["Open file and initialize csv.DictReader"]
    OpenFile --> Loop1{"For each row in reader?"}

    Loop1 -- Yes --> Extract["Extract port and fob from row"]
    Extract --> CheckCond{"if port and fob?"}

    CheckCond -- Yes --> Accumulate["port_fob_totals[port] += float(fob)"]
    Accumulate --> Loop1

    CheckCond -- No --> Loop1

    Loop1 -- No (Loop Finished) --> PrintHeader["Print Header: STEP 1: PORT FOB TOTALS"]
    PrintHeader --> Loop2{"For port_name, total_fob in port_fob_totals.items()?"}

    Loop2 -- Yes --> PrintRow["Print formatted Port and Total FOB"]
    PrintRow --> Loop2

    Loop2 -- No (Loop Finished) --> End([End])
```


In [ ]:
# A POSSIBLE SOLUTION IMPLEMENTING THE WHOLE SEQUENCE OF STEPS

import csv

file_path = 'trase_global_coffee_supply_chain.csv'

# -------------------------------------------------------------------
# STEP 1: `for-in` loop on file iterator to map ports & accumulate total FOB
# -------------------------------------------------------------------
port_fob_totals = {}

with open(file_path, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        port = row['port_of_export_name']
        fob = row['fob']
        if port and fob:
            port_fob_totals[port] = port_fob_totals.get(port, 0.0) + float(fob)

print("=== STEP 1: PORT FOB TOTALS ===")
for port_name, total_fob in port_fob_totals.items():
    print(f"Port: {port_name:<30} | Total FOB: ${total_fob:,.2f}")


# -------------------------------------------------------------------
# STEP 2: `if` statement to narrow records > $50,000 FOB valuation
# -------------------------------------------------------------------
high_value_shipments = []

with open(file_path, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        if row['fob'] and float(row['fob']) > 50000.0:
            high_value_shipments.append(row)

print(f"\n=== STEP 2: HIGH-VALUE FILTER ===")
print(f"Isolated {len(high_value_shipments)} shipments exceeding $50,000 FOB.")


# -------------------------------------------------------------------
# STEP 3: `match-case` pattern matching on HS6 product codes
# -------------------------------------------------------------------
for row in high_value_shipments:
    match row['hs6']:
        case '090111' | '90111':
            category = "Unroasted Green Coffee Beans"
        case '090112' | '90112':
            category = "Decaffeinated Coffee Beans"
        case '210111' | '210111.0':
            category = "Instant Coffee Extracts & Derivatives"
        case _:
            category = "Other Processed Coffee Derivative"
    
    row['product_category'] = category

print("\n=== STEP 3: PRODUCT TYPE CLASSIFICATION ===")
for row in high_value_shipments:
    print(f"HS {row['hs6']} ({row['product_category']}) | FOB: ${float(row['fob']):,.2f} -> Destination: {row['country_of_first_import']}")


# -------------------------------------------------------------------
# STEP 4: `elif` decision chain for regional compliance protocols
# -------------------------------------------------------------------
for row in high_value_shipments:
    bloc = row['country_of_first_import_economic_bloc']
    
    if bloc == 'EUROPEAN UNION':
        protocol = "EUDR Geolocation & Deforestation Audit"
    elif bloc == 'UNITED STATES':
        protocol = "US CBP Agriculture Inspection"
    elif bloc == 'CHINA (MAINLAND)':
        protocol = "GACC Custom Hygiene Clearance"
    else:
        protocol = "Standard MFN Customs Verification"
        
    row['audit_protocol'] = protocol

print("\n=== STEP 4: REGULATORY COMPLIANCE TIERS ===")
for row in high_value_shipments:
    print(f"Destination: {row['country_of_first_import']:<15} | Economic Bloc: {row['country_of_first_import_economic_bloc']:<15} | Protocol: {row['audit_protocol']}")


# -------------------------------------------------------------------
# STEP 5: `while` loop to accumulate audit sample up to 100 raw tons
# -------------------------------------------------------------------
audit_sample_batch = []
accumulated_raw_mass = 0.0
target_quota_tons = 100.0
index = 0

while accumulated_raw_mass < target_quota_tons and index < len(high_value_shipments):
    current_shipment = high_value_shipments[index]
    raw_mass = float(current_shipment['mass_tonnes_raw_equivalent'])
    
    accumulated_raw_mass += raw_mass
    audit_sample_batch.append(current_shipment)
    index += 1

print(f"\n=== STEP 5: AUDIT SAMPLE ACCUMULATION ===")
print(f"Selected {len(audit_sample_batch)} shipments totaling {accumulated_raw_mass:.2f} raw equivalent metric tons.")


# -------------------------------------------------------------------
# STEP 6: `if-else` verification of quota completion
# -------------------------------------------------------------------
print("\n=== STEP 6: QUOTA VERIFICATION ===")
if accumulated_raw_mass >= target_quota_tons:
    print(f"STATUS: SUCCESS. Target sampling quota of {target_quota_tons} tons met.")
    print("Final Audit Sample Breakdown:")
    for item in audit_sample_batch:
        print(f"  - Exporter: {item['exporter_name'][:25]:<25} | Port: {item['port_of_export_name']:<15} | Raw Tons: {float(item['mass_tonnes_raw_equivalent']):6.2f} t | Protocol: {item['audit_protocol']}")
else:
    print(f"STATUS: SHORTFALL. Collected {accumulated_raw_mass:.2f} tons out of required {target_quota_tons} tons.")